# URBAN PULSE – Analyst Investigations

This notebook contains the SQL investigations for the NYC 311 service-request dataset.
It connects to the `urban_pulse_db` MySQL database, retrieves analysis results and creates
charts for the project.

Run the cells from top to bottom. Keep this notebook inside the `notebooks` folder.

**Database security:** Set the `URBAN_PULSE_DB_PASSWORD` environment variable on your
computer before running the connection cell. Do not save database passwords in notebooks
or upload them to GitHub.

In [ ]:
import os
import pandas as pd
import mysql.connector
import matplotlib.pyplot as plt

os.makedirs("../visualizations", exist_ok=True)

In [ ]:
db_password = os.getenv("URBAN_PULSE_DB_PASSWORD")

if not db_password:
    raise ValueError(
        "Set the URBAN_PULSE_DB_PASSWORD environment variable before running this notebook."
    )

connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password=URBAN_PULSE_DB_PASSWORD,
    database="urban_pulse_db"
)

print("MySQL connection successful.")

## Investigation 1 – Top complaint categories

In [ ]:
query = """
SELECT
    complaint_type,
    COUNT(*) AS total_complaints
FROM service_requests
GROUP BY complaint_type
ORDER BY total_complaints DESC
LIMIT 10;
"""

df = pd.read_sql(query, connection)

print(df)

In [ ]:
plt.figure(figsize=(10, 6))

plt.barh(
    df["complaint_type"],
    df["total_complaints"]
)

plt.xlabel("Number of Complaints")
plt.ylabel("Complaint Category")
plt.title("Top 10 Complaint Categories")

plt.gca().invert_yaxis()

plt.tight_layout()
plt.savefig("../visualizations/analyst_chart_3.png", dpi=300, bbox_inches="tight")
plt.show()

## Investigation 2 – Agency workload distribution

In [ ]:
query = """
SELECT
    agency,
    COUNT(*) AS total_complaints
FROM service_requests
GROUP BY agency
ORDER BY total_complaints DESC;
"""

agency_df = pd.read_sql(query, connection)

print(agency_df)

In [ ]:
plt.figure(figsize=(10, 6))

plt.barh(
    agency_df["agency"],
    agency_df["total_complaints"]
)

plt.xlabel("Number of Complaints")
plt.ylabel("Agency")
plt.title("Agency Workload Distribution")

plt.gca().invert_yaxis()

plt.tight_layout()
plt.savefig("../visualizations/analyst_chart_6.png", dpi=300, bbox_inches="tight")
plt.show()

## Investigation 3 – Complaint volume by borough

In [ ]:
query = """
SELECT
    borough,
    COUNT(*) AS total_complaints
FROM service_requests
GROUP BY borough
ORDER BY total_complaints DESC;
"""

borough_df = pd.read_sql(query, connection)

print(borough_df)

In [ ]:
plt.figure(figsize=(10, 6))

plt.barh(
    borough_df["borough"],
    borough_df["total_complaints"]
)

plt.xlabel("Number of Complaints")
plt.ylabel("Borough")
plt.title("311 Service Requests by Borough")

plt.gca().invert_yaxis()

plt.tight_layout()
plt.savefig("../visualizations/analyst_chart_9.png", dpi=300, bbox_inches="tight")
plt.show()

## Investigation 4 – Monthly complaint trend

In [ ]:
query = """
SELECT
    DATE_FORMAT(created_date, '%Y-%m') AS request_month,
    COUNT(*) AS total_complaints
FROM service_requests
WHERE created_date IS NOT NULL
GROUP BY DATE_FORMAT(created_date, '%Y-%m')
ORDER BY request_month;
"""

monthly_df = pd.read_sql(query, connection)

print(monthly_df)

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    monthly_df["request_month"],
    monthly_df["total_complaints"],
    marker="o"
)

plt.xlabel("Month")
plt.ylabel("Number of Complaints")
plt.title("Monthly 311 Service Request Trend")

plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig("../visualizations/analyst_chart_12.png", dpi=300, bbox_inches="tight")
plt.show()

## Investigation 5 – Complaint demand by weekday

In [ ]:
query = """
SELECT
    created_weekday,
    COUNT(*) AS total_complaints
FROM service_requests
GROUP BY created_weekday
ORDER BY total_complaints DESC;
"""

weekday_df = pd.read_sql(query, connection)

print(weekday_df)

In [ ]:
plt.figure(figsize=(10, 5))

plt.barh(
    weekday_df["created_weekday"],
    weekday_df["total_complaints"]
)

plt.xlabel("Number of Complaints")
plt.ylabel("Weekday")
plt.title("311 Service Requests by Weekday")

plt.gca().invert_yaxis()

plt.tight_layout()
plt.savefig("../visualizations/analyst_chart_15.png", dpi=300, bbox_inches="tight")
plt.show()

## Investigation 6 – Complaint demand by hour

In [ ]:
query = """
SELECT
    created_hour,
    COUNT(*) AS total_complaints
FROM service_requests
GROUP BY created_hour
ORDER BY created_hour;
"""

hourly_df = pd.read_sql(query, connection)

print(hourly_df)

In [ ]:
plt.figure(figsize=(12, 5))

plt.bar(
    hourly_df["created_hour"],
    hourly_df["total_complaints"]
)

plt.xticks(range(24))
plt.xlabel("Hour of Day (24-hour format)")
plt.ylabel("Number of Complaints")
plt.title("Complaint Distribution by Hour")

plt.tight_layout()
plt.savefig("../visualizations/hourly_request_distribution.png", dpi=300)
plt.show()

## Investigation 7 – Recorded request status distribution

In [ ]:
query = """
SELECT
    status,
    COUNT(*) AS total_complaints
FROM service_requests
GROUP BY status
ORDER BY total_complaints DESC;
"""

status_df = pd.read_sql(query, connection)

print(status_df)

In [ ]:
plt.figure(figsize=(10, 5))

plt.bar(
    status_df["status"],
    status_df["total_complaints"]
)

plt.xlabel("Request Status")
plt.ylabel("Number of Complaints")
plt.title("Distribution of Recorded Request Statuses")

plt.xticks(rotation=30)
plt.tight_layout()

plt.savefig(
    "../visualizations/request_status_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## Investigation 8 – Resolution-time descriptive statistics

In [ ]:
query = """
SELECT
    COUNT(resolution_hours) AS valid_records,
    AVG(resolution_hours) AS average_hours,
    MIN(resolution_hours) AS minimum_hours,
    MAX(resolution_hours) AS maximum_hours,
    STDDEV_POP(resolution_hours) AS standard_deviation,
    VAR_POP(resolution_hours) AS variance
FROM service_requests
WHERE resolution_hours IS NOT NULL;
"""

resolution_stats_df = pd.read_sql(query, connection)

print(resolution_stats_df)

## Investigation 9 – Median resolution time by complaint category

In [ ]:
# Investigation 9: Median resolution duration by complaint category

median_query = """
WITH ranked_resolution AS (
    SELECT
        complaint_type,
        resolution_hours,
        ROW_NUMBER() OVER (
            PARTITION BY complaint_type
            ORDER BY resolution_hours
        ) AS row_num,
        COUNT(*) OVER (
            PARTITION BY complaint_type
        ) AS category_count
    FROM service_requests
    WHERE resolution_hours IS NOT NULL
)

SELECT
    complaint_type,
    MAX(category_count) AS valid_records,
    AVG(resolution_hours) AS median_resolution_hours
FROM ranked_resolution
WHERE row_num IN (
    FLOOR((category_count + 1) / 2),
    FLOOR((category_count + 2) / 2)
)
GROUP BY complaint_type
ORDER BY median_resolution_hours DESC;
"""

median_df = pd.read_sql(median_query, connection)

print(median_df.head(10))
print("Total categories:", len(median_df))

In [ ]:
# Select top 10 complaint categories by median resolution time
top_10_median = median_df.head(10).copy()

# Create horizontal bar chart
plt.figure(figsize=(12, 7))

plt.barh(
    top_10_median["complaint_type"],
    top_10_median["median_resolution_hours"]
)

plt.xlabel("Median Resolution Time (Hours)")
plt.ylabel("Complaint Category")
plt.title("Top 10 Complaint Categories by Median Resolution Time")

plt.gca().invert_yaxis()

plt.tight_layout()
plt.savefig("../visualizations/analyst_chart_22.png", dpi=300, bbox_inches="tight")
plt.show()

## Investigation 10 – Median resolution time by agency

In [ ]:
# Investigation 10: Median resolution duration by agency

agency_median_query = """
WITH ranked_resolution AS (
    SELECT
        agency,
        resolution_hours,

        ROW_NUMBER() OVER (
            PARTITION BY agency
            ORDER BY resolution_hours
        ) AS row_num,

        COUNT(*) OVER (
            PARTITION BY agency
        ) AS agency_count

    FROM service_requests
    WHERE resolution_hours IS NOT NULL
      AND agency IS NOT NULL
)

SELECT
    agency,
    MAX(agency_count) AS valid_records,
    AVG(resolution_hours) AS median_resolution_hours

FROM ranked_resolution

WHERE row_num IN (
    FLOOR((agency_count + 1) / 2),
    FLOOR((agency_count + 2) / 2)
)

GROUP BY agency
ORDER BY median_resolution_hours DESC;
"""

agency_median_df = pd.read_sql(
    agency_median_query,
    connection
)

print(agency_median_df)

In [ ]:
# Investigation 10: Visualize median resolution duration by agency

plt.figure(figsize=(12, 7))

plt.barh(
    agency_median_df["agency"],
    agency_median_df["median_resolution_hours"]
)

plt.xlabel("Median Resolution Time (Hours)")
plt.ylabel("Agency")
plt.title("Median Resolution Duration by Agency")

plt.gca().invert_yaxis()

plt.tight_layout()

plt.savefig(
    "../visualizations/median_resolution_by_agency.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## Investigation 11 – IQR outlier analysis

In [ ]:
# Investigation 11: Outlier count using IQR

outlier_count_query = """
WITH quartiles AS (
    SELECT
        resolution_hours,
        NTILE(4) OVER (
            ORDER BY resolution_hours
        ) AS quartile_group
    FROM service_requests
    WHERE resolution_hours IS NOT NULL
),

quartile_values AS (
    SELECT
        MAX(
            CASE WHEN quartile_group = 1
            THEN resolution_hours END
        ) AS q1,

        MAX(
            CASE WHEN quartile_group = 3
            THEN resolution_hours END
        ) AS q3
    FROM quartiles
),

limits AS (
    SELECT
        q1 - 1.5 * (q3 - q1) AS lower_limit,
        q3 + 1.5 * (q3 - q1) AS upper_limit
    FROM quartile_values
)

SELECT
    COUNT(*) AS total_valid_records,

    SUM(CASE WHEN resolution_hours < lower_limit
        THEN 1 ELSE 0 END) AS lower_outliers,

    SUM(CASE WHEN resolution_hours > upper_limit
        THEN 1 ELSE 0 END) AS upper_outliers,

    SUM(CASE WHEN resolution_hours < lower_limit
        OR resolution_hours > upper_limit
        THEN 1 ELSE 0 END) AS total_outliers

FROM service_requests
CROSS JOIN limits
WHERE resolution_hours IS NOT NULL;
"""

outlier_count_df = pd.read_sql(
    outlier_count_query,
    connection
)

print(outlier_count_df)

In [ ]:
# Investigation 11: Visualize resolution-time outliers

outlier_count = int(outlier_count_df.loc[0, "total_outliers"])

valid_count = int(outlier_count_df.loc[0, "total_valid_records"])

normal_count = valid_count - outlier_count

plt.figure(figsize=(8, 6))

plt.bar(
    ["Within IQR Limits", "Outliers"],
    [normal_count, outlier_count]
)

plt.ylabel("Number of Records")
plt.title("Resolution Duration: Normal Records vs Outliers")

plt.tight_layout()

plt.savefig(
    "../visualizations/resolution_time_outlier_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## Investigation 12 – Resolution-time distribution and skewness

In [ ]:
# Investigation 12: Resolution-time distribution

distribution_query = """
SELECT
    COUNT(resolution_hours) AS valid_records,
    AVG(resolution_hours) AS mean_hours,
    MIN(resolution_hours) AS minimum_hours,
    MAX(resolution_hours) AS maximum_hours,
    STDDEV_POP(resolution_hours) AS standard_deviation,
    VAR_POP(resolution_hours) AS variance
FROM service_requests
WHERE resolution_hours IS NOT NULL;
"""

distribution_df = pd.read_sql(
    distribution_query,
    connection
)

print(distribution_df)

In [ ]:
# Investigation 12: Calculate resolution-time skewness

skewness_value = pd.read_sql(
    """
    SELECT resolution_hours
    FROM service_requests
    WHERE resolution_hours IS NOT NULL
    """,
    connection
)["resolution_hours"].skew()

print("Skewness:", skewness_value)

if skewness_value > 0:
    print("Distribution is positively skewed (right-skewed).")
elif skewness_value < 0:
    print("Distribution is negatively skewed (left-skewed).")
else:
    print("Distribution is approximately symmetrical.")

In [ ]:
# Investigation 12: Resolution-time distribution histogram

resolution_data = pd.read_sql(
    """
    SELECT resolution_hours
    FROM service_requests
    WHERE resolution_hours IS NOT NULL
    """,
    connection
)

plt.figure(figsize=(12, 6))

plt.hist(
    resolution_data["resolution_hours"],
    bins=50
)

plt.xlabel("Resolution Time (Hours)")
plt.ylabel("Number of Complaints")
plt.title("Distribution of Complaint Resolution Time")

plt.tight_layout()

plt.savefig(
    "../visualizations/resolution_time_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## Investigation 13 – Geographic complaint concentration

In [ ]:
# Investigation 13: Geographic complaint concentration

borough_geo_query = """
SELECT
    borough,
    COUNT(*) AS total_complaints,
    COUNT(latitude) AS records_with_latitude,
    COUNT(longitude) AS records_with_longitude,
    COUNT(
        CASE
            WHEN latitude IS NOT NULL
             AND longitude IS NOT NULL
            THEN 1
        END
    ) AS records_with_coordinates
FROM service_requests
GROUP BY borough
ORDER BY total_complaints DESC;
"""

borough_geo_df = pd.read_sql(
    borough_geo_query,
    connection
)

print(borough_geo_df)

In [ ]:
# Investigation 13: Borough-wise complaint concentration

borough_plot_df = borough_geo_df[
    borough_geo_df["borough"].isin([
        "BROOKLYN",
        "QUEENS",
        "MANHATTAN",
        "BRONX",
        "STATEN ISLAND"
    ])
].copy()

plt.figure(figsize=(10, 6))

plt.barh(
    borough_plot_df["borough"],
    borough_plot_df["total_complaints"]
)

plt.xlabel("Number of Complaints")
plt.ylabel("Borough")
plt.title("Geographic Complaint Concentration by Borough")

plt.gca().invert_yaxis()

plt.tight_layout()

plt.savefig(
    "../visualizations/geographic_complaint_concentration.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## Investigation 14 – Recorded closure rate by agency

In [ ]:
# Investigation 14: Recorded closure rate by agency

closure_query = """
SELECT
    agency,
    COUNT(*) AS total_requests,
    SUM(is_closed) AS closed_requests,
    COUNT(*) - SUM(is_closed) AS not_closed_requests,
    ROUND(
        (SUM(is_closed) / COUNT(*)) * 100,
        2
    ) AS closure_rate_percentage
FROM service_requests
WHERE agency IS NOT NULL
GROUP BY agency
ORDER BY closure_rate_percentage DESC;
"""

closure_df = pd.read_sql(
    closure_query,
    connection
)

print(closure_df)

In [ ]:
# Investigation 14: Agency closure-rate comparison

plt.figure(figsize=(12, 7))

plt.barh(
    closure_df["agency"],
    closure_df["closure_rate_percentage"]
)

plt.xlabel("Recorded Closure Rate (%)")
plt.ylabel("Agency")
plt.title("Recorded Complaint Closure Rate by Agency")

plt.xlim(0, 105)

plt.gca().invert_yaxis()

plt.tight_layout()

plt.savefig(
    "../visualizations/agency_closure_rate_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## Investigation 15 – High-volume complaints with long resolution times

In [ ]:
# Investigation 15: High-volume complaints with long resolution times

high_volume_query = """
WITH ranked_complaints AS (
    SELECT
        complaint_type,
        resolution_hours,

        ROW_NUMBER() OVER (
            PARTITION BY complaint_type
            ORDER BY resolution_hours
        ) AS row_num,

        COUNT(*) OVER (
            PARTITION BY complaint_type
        ) AS category_count

    FROM service_requests
    WHERE resolution_hours IS NOT NULL
      AND complaint_type IS NOT NULL
),

category_medians AS (
    SELECT
        complaint_type,
        MAX(category_count) AS valid_records,
        AVG(resolution_hours) AS median_resolution_hours

    FROM ranked_complaints

    WHERE row_num IN (
        FLOOR((category_count + 1) / 2),
        FLOOR((category_count + 2) / 2)
    )

    GROUP BY complaint_type
),

category_volumes AS (
    SELECT
        complaint_type,
        COUNT(*) AS total_complaints

    FROM service_requests
    WHERE complaint_type IS NOT NULL

    GROUP BY complaint_type
)

SELECT
    v.complaint_type,
    v.total_complaints,
    m.median_resolution_hours

FROM category_volumes v

JOIN category_medians m
    ON v.complaint_type = m.complaint_type

ORDER BY
    v.total_complaints DESC,
    m.median_resolution_hours DESC

LIMIT 20;
"""

high_volume_df = pd.read_sql(
    high_volume_query,
    connection
)

print(high_volume_df)

In [ ]:
# Sort the 20 categories by median resolution time
chart_df = high_volume_df.sort_values(
    by="median_resolution_hours",
    ascending=True
)

# Create the horizontal bar chart
plt.figure(figsize=(12, 8))

plt.barh(
    chart_df["complaint_type"],
    chart_df["median_resolution_hours"]
)

plt.title("Median Resolution Time for High-Volume Complaints")
plt.xlabel("Median Resolution Time (Hours)")
plt.ylabel("Complaint Category")

plt.tight_layout()

# Save the chart
plt.savefig(
    "../visualizations/high_volume_complaints_long_resolution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## Investigation summary

These investigations examine complaint demand, categories, agencies, boroughs, time patterns,
recorded statuses, resolution durations, outliers, geographic concentration and closure rates.

The results describe the selected dataset. Complaint counts and recorded closure rates should
not be treated as direct measures of service quality or citizen satisfaction.